## 1️⃣ Import Libraries
Import libraries required for comprehensive pathway enrichment analysis
using three complementary approaches:
- **gseapy** — Over-Representation Analysis (ORA) and pre-ranked GSEA
  against GO, KEGG, Reactome, and MSigDB gene sets
- **decoupler** — Pathway and transcription factor activity scoring
  using PROGENy and CollecTRI databases
- **pandas/numpy** — data manipulation
- **matplotlib/seaborn** — visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
import decoupler as dc
import os
import warnings
warnings.filterwarnings('ignore')

# Output directories
PATHWAY_DIR = "../output/pathways"
ORA_DIR     = os.path.join(PATHWAY_DIR, "ORA")
GSEA_DIR    = os.path.join(PATHWAY_DIR, "GSEA")
DC_DIR      = os.path.join(PATHWAY_DIR, "decoupler")

for d in [PATHWAY_DIR, ORA_DIR, GSEA_DIR, DC_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Output directories created")
print(f"  {ORA_DIR}")
print(f"  {GSEA_DIR}")
print(f"  {DC_DIR}")

import importlib.metadata
print(f"\ngseapy version   : {importlib.metadata.version('gseapy')}")
print(f"decoupler version: {importlib.metadata.version('decoupler')}")

## 2️⃣ Load DEG Results
Load the master DEG table generated in Script 05 containing
differential expression results for all 18 cell types.
Significant DEGs (FDR<0.05, |log2FC|>0.25) are used for ORA,
while all genes ranked by log2FC are used for pre-ranked GSEA.

In [ ]:
# Load master DEG table
master_deg = pd.read_csv("../output/DEG_master_table.csv")

# Significant DEGs for ORA
sig_deg = master_deg[master_deg['significant'] == True].copy()

print(f"Total genes tested   : {len(master_deg):,}")
print(f"Significant DEGs     : {len(sig_deg):,}")
print(f"Cell types with DEGs : {sig_deg['cell_type'].nunique()}")
print(f"\nDEGs per cell type:")
print(sig_deg.groupby('cell_type').size().sort_values(ascending=False))

## 3️⃣ Over-Representation Analysis (ORA)
Test whether significant DEGs are enriched in known biological pathways
using three databases simultaneously:
- **GO Biological Process** — fundamental biological processes
- **KEGG** — metabolic and signaling pathways
- **Reactome** — detailed molecular pathway maps

### Method:
ORA uses a hypergeometric test to ask:
> *"Are more genes from my DEG list in this pathway than expected by chance?"*

Analysis is performed for each cell type separately using only
upregulated genes (log2FC > 0.25) to identify activated pathways in T1D.

In [ ]:
# Databases to query
databases = ['GO_Biological_Process_2023',
             'KEGG_2021_Human',
             'Reactome_2022']

# Cell types with enough DEGs
celltypes_ora = sig_deg.groupby('cell_type').size()
celltypes_ora = celltypes_ora[celltypes_ora >= 10].index.tolist()

ora_results = {}

for ct in sorted(celltypes_ora):
    print(f"\n▶ ORA: {ct}")

    # Get upregulated genes
    up_genes = sig_deg[
        (sig_deg['cell_type'] == ct) &
        (sig_deg['logfoldchanges'] > 0.25)
    ]['names'].tolist()

    if len(up_genes) < 10:
        print(f"  ⚠️ Not enough upregulated genes → skipping")
        continue

    print(f"  Upregulated genes: {len(up_genes)}")

    try:
        enr = gp.enrichr(
            gene_list=up_genes,
            gene_sets=databases,
            organism='human',
            outdir=None,
            verbose=False
        )

        results = enr.results
        results['cell_type'] = ct

        # Save per cell type
        ct_clean = ct.replace('/', '_').replace(' ', '_')
        out_file = os.path.join(ORA_DIR, f"ORA_{ct_clean}.csv")
        results.to_csv(out_file, index=False)

        ora_results[ct] = results
        sig_paths = results[results['Adjusted P-value'] < 0.05]
        print(f"  Significant pathways: {len(sig_paths)}")
        print(f"  ✔ Saved: {out_file}")

    except Exception as e:
        print(f"  ❌ Error: {e}")

print("\n🎉 ORA complete!")

## 4️⃣ Pre-Ranked GSEA
Gene Set Enrichment Analysis using all genes ranked by log2 fold
change — more powerful than ORA as it uses the full gene list,
not just significant DEGs.

### Method:
- Genes ranked by log2FC (high = upregulated in T1D)
- Tests whether pathway genes cluster at top or bottom of ranking
- **Positive NES** — pathway upregulated in T1D
- **Negative NES** — pathway downregulated in T1D

### Databases:
- **MSigDB Hallmarks** — 50 curated hallmark gene sets
- **Reactome** — detailed molecular pathways

### Ranking Metric:
Instead of raw log2FC, genes are ranked by **signed -log10(p-value) × log2FC**
to avoid ties caused by zero p-values in Wilcoxon results.
This produces a more informative ranking for GSEA.

In [ ]:
# Databases for GSEA
gsea_databases = ['MSigDB_Hallmark_2020', 'Reactome_2022']

# Cell types with enough genes
celltypes_gsea = master_deg.groupby('cell_type').size()
celltypes_gsea = celltypes_gsea[celltypes_gsea >= 100].index.tolist()

gsea_results = {}

for ct in sorted(celltypes_gsea):
    print(f"\n▶ GSEA: {ct}")

    # Get all genes
    ct_deg = master_deg[master_deg['cell_type'] == ct].copy()
    ct_deg = ct_deg.dropna(subset=['logfoldchanges', 'pvals'])
    ct_deg = ct_deg.drop_duplicates(subset='names')

    # Better ranking metric — signed -log10(pval) × direction
    ct_deg['rank_metric'] = (
        np.sign(ct_deg['logfoldchanges']) *
        -np.log10(ct_deg['pvals'] + 1e-300)
    )

    # Sort by ranking metric
    ct_deg = ct_deg.sort_values('rank_metric', ascending=False)
    ranked = ct_deg.set_index('names')['rank_metric']

    print(f"  Genes ranked: {len(ranked)}")
    print(f"  Rank range  : {ranked.max():.2f} to {ranked.min():.2f}")

    try:
        pre_res = gp.prerank(
            rnk=ranked,
            gene_sets=gsea_databases,
            organism='human',
            outdir=None,
            min_size=10,
            max_size=500,
            permutation_num=1000,
            verbose=False,
            seed=42
        )

        results = pre_res.res2d
        results['cell_type'] = ct

        # Save
        ct_clean = ct.replace('/', '_').replace(' ', '_')
        out_file = os.path.join(GSEA_DIR, f"GSEA_{ct_clean}.csv")
        results.to_csv(out_file, index=False)

        gsea_results[ct] = results
        sig_paths = results[results['FDR q-val'] < 0.25]
        print(f"  Significant pathways (FDR<0.25): {len(sig_paths)}")
        print(f"  ✔ Saved: {out_file}")

    except Exception as e:
        print(f"  ❌ Error: {e}")

print("\n🎉 GSEA complete!")

## 5️⃣ Pathway Activity Scoring with PROGENy (decoupler)
Estimate activity of 14 key biological pathways using PROGENy
via decoupler on pseudobulk data.

### Why Pseudobulk for PROGENy?
Running PROGENy on all 29,287 individual cells requires too much
memory. Instead we aggregate counts per sample per cell type
(pseudobulk) — reducing the data to ~720 pseudobulk samples.
This is also more statistically robust as pseudobulk better
represents biological replicates.

### PROGENy Pathways:
Androgen, EGFR, Estrogen, Hypoxia, JAK-STAT, MAPK, NFkB, PI3K,
TGFb, TNFa, Trail, VEGF, WNT, p53

In [ ]:
import scanpy as sc
import decoupler as dc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

PATHWAY_DIR = "../output/pathways"
os.makedirs(PATHWAY_DIR, exist_ok=True)

# Load in backed mode
print("⏳ Loading data...")
adata = sc.read_h5ad(
    "../data/processed/GSE279086_annotated.h5ad",
    backed='r'
)
print(f"✅ Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# Create pseudobulk
print("\n⏳ Creating pseudobulk...")
pdata = dc.pp.pseudobulk(
    adata,
    sample_col='sample',
    groups_col='majority_voting',
    layer='counts',
    mode='sum'
)
print(f"✅ Pseudobulk: {pdata.shape[0]} samples × {pdata.shape[1]} genes")
del adata

# Normalize
print("\n⏳ Normalizing...")
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
print("✅ Normalization complete")

# Get PROGENy network
print("\n⏳ Getting PROGENy network...")
progeny = dc.op.progeny(organism='human', top=500)
print(f"Pathways: {sorted(progeny['source'].unique())}")

# Run PROGENy MLM — capture returned object
print("\n⏳ Running PROGENy MLM...")
pdata_fixed = dc.mt.mlm(pdata, progeny, tmin=3, verbose=True)

# Check obsm keys on returned object
print("\nObsm keys:", list(pdata_fixed.obsm.keys()))

# Extract scores
progeny_scores = pdata_fixed.obsm['score_mlm'].copy()
progeny_pvals  = pdata_fixed.obsm['padj_mlm'].copy()

# Add metadata
progeny_scores['cell_type'] = pdata_fixed.obs['majority_voting'].values
progeny_scores['disease']   = pdata_fixed.obs['disease'].values
progeny_scores['sample']    = pdata_fixed.obs['sample'].values

print(f"\n✅ PROGENy complete")
print(f"Shape    : {progeny_scores.shape}")
print(f"Pathways : {[c for c in progeny_scores.columns if c not in ['cell_type','disease','sample']]}")

# Save
progeny_scores.to_csv(f"{PATHWAY_DIR}/PROGENy_scores.csv", index=True)
progeny_pvals.to_csv(f"{PATHWAY_DIR}/PROGENy_pvals.csv", index=True)
print(f"\n✅ Saved: {PATHWAY_DIR}/PROGENy_scores.csv")
print(f"✅ Saved: {PATHWAY_DIR}/PROGENy_pvals.csv")

## 6️⃣ Visualize PROGENy Pathway Activity
Generate a heatmap showing mean pathway activity scores per cell type
and disease condition to identify dysregulated pathways in T1D.

In [ ]:
# Calculate mean pathway scores per cell type and disease
pathway_cols = [c for c in progeny_scores.columns
                if c not in ['cell_type', 'disease', 'sample']]

mean_scores = progeny_scores.groupby(
    ['cell_type', 'disease'])[pathway_cols].mean().reset_index()

# Pivot for heatmap — T1D vs Control per cell type
t1d     = mean_scores[mean_scores['disease'] == 'Type 1 Diabetes']\
          .set_index('cell_type')[pathway_cols]
control = mean_scores[mean_scores['disease'] == 'Lean Control']\
          .set_index('cell_type')[pathway_cols]

# Difference: T1D - Control
diff = t1d - control
diff = diff.dropna()

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    diff,
    cmap='RdBu_r',
    center=0,
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    ax=ax
)
ax.set_title(
    'PROGENy Pathway Activity\n(T1D − Lean Control per Cell Type)',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel("Pathway")
ax.set_ylabel("Cell Type")
plt.tight_layout()
plt.savefig("../figures/PROGENy_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/PROGENy_heatmap.png")

## 7️⃣ Visualize ORA Results — Top Pathways Per Cell Type
Generate a dot plot showing the top enriched pathways from ORA
(GO, KEGG, Reactome) for the most affected cell types.

In [ ]:
# Load and combine all ORA results
ora_files = [f for f in os.listdir("../output/pathways/ORA") if f.endswith('.csv')]

ora_all = []
for f in ora_files:
    df = pd.read_csv(f"../output/pathways/ORA/{f}")
    ora_all.append(df)

ora_combined = pd.concat(ora_all, ignore_index=True)

# Filter significant
ora_sig = ora_combined[ora_combined['Adjusted P-value'] < 0.05].copy()
ora_sig['-log10(padj)'] = -np.log10(ora_sig['Adjusted P-value'] + 1e-300)

print(f"Total significant pathways : {len(ora_sig):,}")
print(f"Cell types                 : {ora_sig['cell_type'].nunique()}")
print(f"Databases                  : {ora_sig['Gene_set'].unique()}")

# Top 3 pathways per cell type per database
top_ora = (
    ora_sig
    .sort_values('-log10(padj)', ascending=False)
    .groupby(['cell_type', 'Gene_set'])
    .head(3)
)

# Plot top 6 cell types
top_cts = ['C-TAL', 'CCD-IC-A', 'PC', 'EC-PTC', 'PT', 'DTL']
plot_df = top_ora[top_ora['cell_type'].isin(top_cts)].copy()
plot_df['Term_short'] = plot_df['Term'].str[:50]

fig, axes = plt.subplots(2, 3, figsize=(24, 16))
axes = axes.flatten()

for idx, ct in enumerate(top_cts):
    ct_df = plot_df[plot_df['cell_type'] == ct]\
            .sort_values('-log10(padj)', ascending=True).tail(15)

    axes[idx].barh(
        ct_df['Term_short'],
        ct_df['-log10(padj)'],
        color='#E74C3C',
        alpha=0.8
    )
    axes[idx].set_title(f"{ct}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("-log10(adj. p-value)")
    axes[idx].tick_params(axis='y', labelsize=7)
    axes[idx].axvline(x=-np.log10(0.05), color='black',
                      linestyle='--', linewidth=0.8)

plt.suptitle("Top ORA Pathways per Cell Type (T1D vs Lean Control)",
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../figures/ORA_top_pathways.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/ORA_top_pathways.png")

## 8️⃣ Visualize GSEA Results
Generate a bar plot showing top enriched pathways from pre-ranked
GSEA for the most affected cell types.
- **Red bars** — activated pathways in T1D (positive NES)
- **Blue bars** — suppressed pathways in T1D (negative NES)

In [ ]:
# Load and combine GSEA results
gsea_files = [f for f in os.listdir("../output/pathways/GSEA")
              if f.endswith('.csv')]

gsea_all = []
for f in gsea_files:
    df = pd.read_csv(f"../output/pathways/GSEA/{f}")
    ct = f.replace('GSEA_', '').replace('.csv', '').replace('_', ' ')
    df['cell_type'] = ct
    gsea_all.append(df)

gsea_combined = pd.concat(gsea_all, ignore_index=True)

# Filter significant
gsea_sig = gsea_combined[gsea_combined['FDR q-val'] < 0.25].copy()
gsea_sig['-log10(FDR)'] = -np.log10(gsea_sig['FDR q-val'] + 1e-300)

# Clean term names
gsea_sig['Term_short'] = gsea_sig['Term']\
    .str.replace('MSigDB_Hallmark_2020__', '', regex=False)\
    .str.replace('Reactome_2022__', '', regex=False)\
    .str[:40]

print(f"Significant GSEA pathways : {len(gsea_sig):,}")
print(f"Cell types                : {gsea_sig['cell_type'].nunique()}")
print(f"Cell type names           : {gsea_sig['cell_type'].unique()}")

# Top 5 pathways per cell type by absolute NES
top_gsea = (
    gsea_sig
    .assign(abs_NES=gsea_sig['NES'].abs())
    .sort_values('abs_NES', ascending=False)
    .groupby('cell_type')
    .head(5)
)

# Plot top 6 cell types — use exact names
top_cts = ['C-TAL', 'CNT', 'aPT', 'EC-DVR', 'IC-B', 'VSMC P']
plot_df = top_gsea[top_gsea['cell_type'].isin(top_cts)].copy()

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.flatten()

for idx, ct in enumerate(top_cts):
    ct_df = plot_df[plot_df['cell_type'] == ct]\
            .sort_values('NES', ascending=True)

    colors = ['#3498DB' if n < 0 else '#E74C3C' for n in ct_df['NES']]

    axes[idx].barh(
        ct_df['Term_short'],
        ct_df['NES'],
        color=colors,
        alpha=0.8
    )
    axes[idx].set_title(f"{ct}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("NES (Normalized Enrichment Score)")
    axes[idx].tick_params(axis='y', labelsize=7)
    axes[idx].axvline(x=0, color='black', linewidth=0.8)

plt.suptitle("Top GSEA Pathways per Cell Type (T1D vs Lean Control)",
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../figures/GSEA_top_pathways.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/GSEA_top_pathways.png")

## 9️⃣ Combined Biological Insights
Synthesize findings from ORA, GSEA and PROGENy into a unified
biological narrative showing the key dysregulated processes
in T1D kidney across all cell types.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(26, 10))

# ── Panel 1: GSEA NES heatmap ─────────────────────────────────
gsea_files = [f for f in os.listdir("../output/pathways/GSEA")
              if f.endswith('.csv')]
gsea_all = []
for f in gsea_files:
    df = pd.read_csv(f"../output/pathways/GSEA/{f}")
    ct = f.replace('GSEA_', '').replace('.csv', '').replace('_', ' ')
    df['cell_type'] = ct
    gsea_all.append(df)
gsea_combined = pd.concat(gsea_all, ignore_index=True)
gsea_sig = gsea_combined[gsea_combined['FDR q-val'] < 0.25].copy()

# Clean term names
gsea_sig['Term_short'] = gsea_sig['Term']\
    .str.replace('MSigDB_Hallmark_2020__', '', regex=False)\
    .str.replace('Reactome_2022__', '', regex=False)\
    .str[:30]

# Top pathway per cell type by abs NES
top1 = gsea_sig\
    .assign(abs_NES=gsea_sig['NES'].abs())\
    .sort_values('abs_NES', ascending=False)\
    .groupby('cell_type').first().reset_index()

gsea_pivot = gsea_sig\
    .assign(abs_NES=gsea_sig['NES'].abs())\
    .sort_values('abs_NES', ascending=False)\
    .drop_duplicates(subset=['cell_type', 'Term_short'])\
    .groupby('cell_type').head(5)\
    .pivot_table(index='cell_type', columns='Term_short',
                 values='NES', aggfunc='mean')\
    .fillna(0)

sns.heatmap(
    gsea_pivot,
    cmap='RdBu_r',
    center=0,
    linewidths=0.3,
    ax=axes[0],
    cbar_kws={'label': 'NES'}
)
axes[0].set_title('GSEA Top Pathways\n(NES per Cell Type)',
                   fontsize=12, fontweight='bold')
axes[0].tick_params(axis='both', labelsize=6)
axes[0].set_xlabel("")
axes[0].set_ylabel("Cell Type")

# ── Panel 2: PROGENy heatmap ──────────────────────────────────
progeny_scores = pd.read_csv("../output/pathways/PROGENy_scores.csv",
                              index_col=0)
pathway_cols = [c for c in progeny_scores.columns
                if c not in ['cell_type', 'disease', 'sample']]

mean_scores = progeny_scores.groupby(
    ['cell_type', 'disease'])[pathway_cols].mean()
t1d     = mean_scores.xs('Type 1 Diabetes', level='disease')
control = mean_scores.xs('Lean Control',     level='disease')
diff    = (t1d - control).dropna()

sns.heatmap(
    diff,
    cmap='RdBu_r',
    center=0,
    linewidths=0.5,
    ax=axes[1],
    cbar_kws={'label': 'T1D - Control'}
)
axes[1].set_title('PROGENy Pathway Activity\n(T1D − Lean Control)',
                   fontsize=12, fontweight='bold')
axes[1].tick_params(axis='both', labelsize=7)
axes[1].set_xlabel("Pathway")
axes[1].set_ylabel("Cell Type")

# ── Panel 3: ORA pathway counts ───────────────────────────────
ora_files = [f for f in os.listdir("../output/pathways/ORA")
             if f.endswith('.csv')]
ora_all = []
for f in ora_files:
    df = pd.read_csv(f"../output/pathways/ORA/{f}")
    ora_all.append(df)
ora_combined = pd.concat(ora_all, ignore_index=True)
ora_sig = ora_combined[ora_combined['Adjusted P-value'] < 0.05]

ora_counts = ora_sig.groupby(['cell_type', 'Gene_set'])\
    .size().unstack(fill_value=0)

ora_counts.plot(
    kind='bar',
    ax=axes[2],
    colormap='Set2',
    width=0.8
)
axes[2].set_title('Significant ORA Pathways\nper Cell Type & Database',
                   fontsize=12, fontweight='bold')
axes[2].set_xlabel("")
axes[2].set_ylabel("Number of Pathways")
axes[2].tick_params(axis='x', rotation=90, labelsize=7)
axes[2].legend(title='Database', fontsize=7)

plt.suptitle(
    "Integrated Pathway Analysis — T1D vs Lean Control\n"
    "GSEA (NES) | PROGENy Pathway Activity | ORA Database Enrichment",
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig("../figures/Combined_pathway_insights.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/Combined_pathway_insights.png")